[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/cours/seance2_cours.ipynb)

# Séance 3.2 — Comparer deux groupes — hasard ou vrai écart ?

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- expliquer pourquoi une moyenne calculée sur un échantillon n'est jamais exacte
- construire un intervalle de confiance à 95 % par rééchantillonnage
- comparer deux groupes avec un test t et lire sa p-value
- distinguer « pas de différence » de « pas de différence détectable »
- repérer les deux pièges qui rendent un test faux : dépendance des observations et tests répétés

## Une décision à 11 euros près

Le budget marketing de l'an prochain va sur **un seul** des deux marchés,
France ou Allemagne. Vous avez les paniers moyens.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

fr = cmd.query("pays == 'France'")["ca"]
de = cmd.query("pays == 'Allemagne'")["ca"]

print("France    :", round(fr.mean(), 2), "euros sur", len(fr), "commandes")
print("Allemagne :", round(de.mean(), 2), "euros sur", len(de), "commandes")

**529,58 € contre 540,29 €.** L'Allemagne gagne de 10,71 €.

Prenez la décision maintenant, mentalement. On y revient à la fin de la
séance.

## 1. Quantifier l'incertitude d'une moyenne

Ces 253 commandes françaises ne sont pas « la France ». C'est **une année de
commandes** parmi toutes celles qui auraient pu se produire. Une autre année,
les mêmes clients auraient commandé un peu autrement, et la moyenne serait
différente.

**De combien ?** C'est exactement ça, quantifier l'incertitude d'une moyenne :
donner l'ordre de grandeur de ce que le hasard de l'échantillon peut faire
bouger. On peut le mesurer sans aucune formule, avec une seule méthode :
**rejouer l'échantillon**.

### L'outil : `sample()`

`sample` tire des observations au hasard dans une colonne.

In [ ]:
# 5 valeurs tirees au hasard parmi les 253 commandes francaises
print(fr.sample(5, replace=True, random_state=0).round(2).tolist())
print(fr.sample(5, replace=True, random_state=1).round(2).tolist())

Trois arguments, et ils comptent tous les trois :

| Argument | Ce qu'il fait |
|---|---|
| le premier nombre | combien d'observations on tire |
| `replace=True` | tirage **avec remise** : une même commande peut sortir deux fois, une autre pas du tout |
| `random_state=0` | fixe le hasard, pour que tout le monde obtienne les mêmes valeurs |

La remise est le point clé. Sans elle, tirer 253 valeurs parmi 253 redonnerait
les 253 mêmes commandes dans le désordre — donc toujours la même moyenne, et
aucune information sur l'incertitude. Avec remise, chaque tirage est **une
année plausible** : les mêmes clients, un déroulé un peu différent.

> ⚠️ `random_state` n'a rien de statistique. C'est un numéro qui rend le
> tirage reproductible : même numéro, mêmes valeurs, sur n'importe quelle
> machine. On le fera varier de 0 à 999 pour obtenir mille tirages différents.

Oublier `replace=True` est l'erreur classique, et pandas la signale. La
cellule suivante est volontairement fausse : elle demande 1 000 valeurs à une
colonne qui n'en contient que 253, sans autoriser la remise.

In [ ]:
fr.sample(1000)   ## erreur volontaire : il manque replace=True

Dernière ligne :

```
ValueError: Cannot take a larger sample than population when 'replace=False'
```

Traduction : sans remise, on ne peut pas tirer plus de valeurs qu'il n'y en a
dans la colonne. **Seule la dernière ligne d'une erreur compte.**

### Mille années plausibles

On tire donc 253 commandes avec remise parmi ces 253-là, on calcule la
moyenne, et on recommence mille fois.

In [ ]:
# 1 000 tirages, 1 000 moyennes : random_state=i change le tirage
boot = pd.Series([fr.sample(len(fr), replace=True, random_state=i).mean()
                  for i in range(1000)])

print("la moyenne francaise varie de", round(boot.min(), 2),
      "a", round(boot.max(), 2))

In [ ]:
boot.plot(kind="hist", bins=40, figsize=(7, 4))   ## la forme du doute
plt.title("1000 moyennes francaises possibles")
plt.xlabel("panier moyen (euros)")
plt.show()

De **411 €** à **731 €**. Les mêmes clients, la même activité — et 320 €
d'écart entre la plus basse et la plus haute des moyennes plausibles.

L'écart franco-allemand que vous vouliez arbitrer était de **11 €**. Il tient
trente fois dans l'incertitude de la seule moyenne française.

## 2. L'intervalle de confiance à 95 %

On garde les **95 % centraux** de ces mille moyennes : on jette les 2,5 % les
plus basses et les 2,5 % les plus hautes.

In [ ]:
bas = boot.quantile(0.025)    ## on jette les 2,5 % les plus basses
haut = boot.quantile(0.975)   ## et les 2,5 % les plus hautes

print("panier moyen francais : entre", round(bas, 2), "et", round(haut, 2))

> 💡 **Comment on le dit.** « Compte tenu de ce qu'on a observé, les valeurs
> plausibles du panier moyen français vont de 445 € à 629 €. » C'est une
> **fourchette de plausibilité**, pas une probabilité sur la vraie valeur.

Et l'Allemagne ?

In [ ]:
bd = pd.Series([de.sample(len(de), replace=True, random_state=i).mean()
                for i in range(1000)])

print("France    :", round(bas, 2), "-", round(haut, 2))
print("Allemagne :", round(bd.quantile(0.025), 2), "-", round(bd.quantile(0.975), 2))

**445–629 contre 461–641.** Les deux fourchettes se recouvrent presque
entièrement. Toute valeur du premier intervalle est plausible pour le second.

Il n'y a rien à arbitrer entre ces deux marchés sur la base du panier moyen.

## 3. Le test t : la même question, en un nombre

Le test répond exactement à la question qu'on vient de se poser :

> **Si les deux marchés étaient identiques, à quelle fréquence observerait-on
> un écart au moins aussi grand que celui-ci ?**

Cette fréquence, c'est la **p-value**.

In [ ]:
# equal_var=False : on ne suppose pas la meme dispersion des deux cotes.
# C'est le choix raisonnable par defaut.
resultat = stats.ttest_ind(fr, de, equal_var=False)

print("p-value :", round(resultat.pvalue, 3))   ## 0,874 : rien a retenir

**p = 0,874.** Si les deux marchés étaient identiques, on observerait un écart
d'au moins 11 € dans **87 % des cas**. Cet écart n'a rien de remarquable.

La convention : on retient un écart quand **p < 0,05**.

| p-value | Ce qu'on en fait |
|---|---|
| p < 0,05 | l'écart serait rare sous l'hypothèse d'égalité : on le retient |
| p ≥ 0,05 | l'écart est compatible avec le hasard : **on ne conclut rien** |

> ⚠️ **La faute à ne pas commettre.** p = 0,874 ne dit **pas** que les deux
> marchés sont identiques. Il dit que ces données ne permettent pas de les
> départager. Avec 5 000 commandes de chaque côté, le même écart de 11 €
> pourrait très bien devenir significatif.
>
> « Pas de différence détectable » et « pas de différence » sont deux
> affirmations différentes.

Le seuil de 0,05 est une **convention**, pas une loi de la nature. Une p-value
de 0,049 et une de 0,051 décrivent à peu près la même situation.

## 4. Un cas où le test tranche — et le piège qui va avec

Même test, Royaume-Uni contre Irlande.

In [ ]:
uk = cmd.query("pays == 'Royaume-Uni'")["ca"]
irl = cmd.query("pays == 'Irlande'")["ca"]

print("p-value :", stats.ttest_ind(uk, irl, equal_var=False).pvalue)   ## minuscule

**p = 0,000000014.** Un écart comme celui-là ne s'explique pas par le hasard.
Le panier irlandais est réellement plus élevé.

Avant d'écrire la recommandation, une question : **combien de personnes** y
a-t-il derrière ces commandes ?

In [ ]:
cmd.groupby("pays")["client_id"].nunique().nlargest(4)   ## des GENS

**Deux clients irlandais.** Contre 235 britanniques.

Le test t suppose que les observations sont **indépendantes** : que chaque
commande apporte une information nouvelle. Ici, 256 commandes viennent de
deux acheteurs. Ce ne sont pas 256 comportements d'achat, ce sont **deux**,
répétés 128 fois chacun en moyenne.

La p-value a été calculée comme s'il y en avait 256. Elle est donc beaucoup
trop optimiste — et elle l'est dans le sens qui vous plaît.

> ⚠️ **Avant tout test : comptez les individus, pas les lignes.** C'est le
> même réflexe `nunique` qu'en séance 2.3, et il a la même importance ici.

## 5. Significatif ne veut pas dire important

Un test dit si un écart existe. Il ne dit rien de sa **taille**, et c'est la
taille qui fait la décision.

In [ ]:
print("ecart :", round(irl.mean() - uk.mean(), 2), "euros")   ## la taille
print("rapport :", round(irl.mean() / uk.mean(), 2), "fois plus")

626 € d'écart, un panier 2,6 fois plus élevé : ici l'effet est énorme *et*
significatif.

Mais l'inverse arrive tout le temps : sur 100 000 commandes, un écart de 3 €
sort avec p < 0,001. Statistiquement indiscutable, commercialement sans
intérêt. **Affichez toujours l'écart en euros à côté de la p-value.**

## 6. Tests répétés : chercher jusqu'à trouver finit toujours par trouver

Revenons sur le seuil de 0,05. On l'a présenté comme une convention ; c'est
aussi, très concrètement, un **taux d'erreur accepté d'avance**.

Relisez la définition : la p-value est la probabilité d'observer un écart au
moins aussi grand que celui-ci **si les deux groupes étaient identiques**.
Retenir tout ce qui passe sous 0,05, c'est donc accepter que, sur des groupes
réellement identiques, **un test sur vingt** annonce quand même une
différence. Ce n'est pas un défaut de la méthode : c'est le contrat qu'on a
signé en choisissant ce seuil.

### L'intuition : que devient ce contrat quand on lance vingt tests ?

Sur des groupes identiques, un test a 95 % de chances de rester silencieux.
Deux tests indépendants : 0,95 × 0,95. Vingt tests : 0,95 puissance 20.

In [ ]:
aucun_faux = 0.95 ** 20   ## les 20 tests restent silencieux
print("probabilite qu'aucun ne crie :", round(100 * aucun_faux, 1), "%")
print("donc au moins un faux positif :", round(100 * (1 - aucun_faux), 1), "%")

**64 %.** En lançant vingt tests sur des groupes entre lesquels il n'y a
strictement rien à trouver, on a **deux chances sur trois** d'en voir sortir
au moins un « significatif ».

Ce chiffre est contre-intuitif dans les deux sens, et ça vaut la peine de
s'arrêter dessus :

- ce n'est pas 5 % — le seuil vaut pour **un** test, pas pour une campagne ;
- ce n'est pas 100 % non plus — rien ne garantit qu'un faux positif sortira,
  c'est une probabilité, pas une fatalité.

Et le chiffre grimpe vite : 5 tests → 23 %, 20 tests → 64 %, 100 tests → 99 %.
Un tableau de bord qui compare tout seul dix segments sur dix indicateurs
lance cent tests par semaine.

### La vérification, sur ce fichier

Vingt comparaisons entre deux moitiés du fichier tirées **au hasard**. Les
deux moitiés sortent de la même population : par construction, il n'y a
**aucune** différence à trouver.

In [ ]:
ps = []
for i in range(20):
    a = cmd.sample(frac=0.5, random_state=i)   ## une moitie au hasard
    b = cmd.drop(a.index)                      ## l'autre moitie
    ps.append(stats.ttest_ind(a["ca"], b["ca"], equal_var=False).pvalue)

ps = pd.Series(ps)
print("plus petite p-value :", round(ps.min(), 4), "| nombre sous 0,05 :", (ps < 0.05).sum())

**Un test sur vingt ressort à p = 0,024** — exactement ce que la probabilité
annonçait, et une différence qui n'existe pas.

Imaginez maintenant qu'on ne vous montre que celui-là. La p-value est juste,
le calcul est correct, le graphique sera convaincant, et la conclusion sera
fausse. **Ce qui manque n'est pas dans le résultat : c'est le nombre de tests
qu'il a fallu lancer pour l'obtenir.**

> ⚠️ **La règle : la question d'abord, le test ensuite.** Une p-value ne vaut
> que pour un test décidé **avant** d'avoir regardé les données. Si vous en
> avez lancé vingt, dites-le — et durcissez le seuil en conséquence. La
> correction la plus simple, dite **de Bonferroni**, divise le seuil par le
> nombre de tests : 0,05 / 20 = 0,0025. Notre p-value de 0,024 ne passe plus
> du tout.

### Alors, France ou Allemagne ?

Aucune des deux, sur ce critère. L'écart de 11 € n'est pas mesurable avec
250 commandes de chaque côté. Il faut arbitrer sur autre chose — le coût
d'acquisition, la marge, la logistique — ou récolter plus de données.

**Dire « ces données ne permettent pas de trancher » est une réponse
professionnelle.** C'est souvent la bonne.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| un rééchantillon | `serie.sample(len(serie), replace=True, random_state=i)` |
| l'intervalle de confiance à 95 % | `boot.quantile(0.025)` et `boot.quantile(0.975)` |
| comparer deux moyennes | `stats.ttest_ind(a, b, equal_var=False)` |
| la p-value seule | `stats.ttest_ind(a, b, equal_var=False).pvalue` |
| compter les individus derrière un groupe | `df.groupby("pays")["client_id"].nunique()` |
| le risque d'au moins un faux positif sur n tests | `1 - 0.95 ** n` |

## Lire une p-value

| p-value | Ce qu'on peut dire |
|---|---|
| **inférieure à 0,05** | l'écart observé serait rare si les deux groupes étaient identiques : on le retient |
| **supérieure à 0,05** | l'écart est compatible avec le hasard : **on ne conclut rien** |

## Les trois phrases à retenir

1. **« On ne peut pas conclure » n'est pas « il n'y a pas de différence ».**
   Avec 253 et 252 commandes, on ne détecte pas un écart de 11 €. Cela ne
   prouve pas qu'il est nul.

2. **Un test suppose des observations indépendantes.** 256 commandes
   irlandaises passées par **2 clients** ne sont pas 256 observations
   indépendantes, quelle que soit la p-value affichée.

3. **Une p-value n'est valable que pour le test qu'on avait prévu de faire.**
   Sur des groupes identiques, vingt tests ont **64 %** de chances d'en sortir
   un « significatif ». Par construction : c'est le seuil lui-même qu'on a
   retrouvé, pas un résultat.